In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")

# 1. Load Dataset
# Note: Ensure raw_sales_data.csv is placed in week_1_analytics/data/
try:
    df = pd.read_csv('../data/raw_sales_data.csv')
    print("Dataset successfully loaded. Shape:", df.shape)
except FileNotFoundError:
    print("Generating synthetic sales dataset for EDA pipeline...")
    np.random.seed(42)
    df = pd.DataFrame({
        'order_id': [f'ORD{1000+i}' for i in range(200)],
        'customer_id': [f'CUST{np.random.randint(100, 120)}' for _ in range(200)],
        'order_date': pd.date_range(start='2026-01-01', periods=200, freq='D'),
        'product': np.random.choice(['Equity Plan', 'F&O Analytics', 'Mutual Fund Advisory', 'API Access'], size=200),
        'revenue': np.random.normal(loc=150, scale=40, size=200).round(2),
        'quantity': np.random.randint(1, 5, size=200)
    })
    # Inject missing values & duplicates for cleaning demonstration
    df.loc[10:15, 'revenue'] = np.nan
    df = pd.concat([df, df.iloc[:5]], ignore_index=True)

# 2. Data Cleaning Routine
print("\n--- Initial Missing Values ---")
print(df.isnull().sum())

# Remove duplicate records
duplicates_count = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f"\nRemoved {duplicates_count} duplicate records.")

# Impute missing numerical values using median
if 'revenue' in df.columns:
    median_rev = df['revenue'].median()
    df['revenue'] = df['revenue'].fillna(median_rev)
    print(f"Imputed missing revenue values with median: ${median_rev:.2f}")

# Format date column
df['order_date'] = pd.to_datetime(df['order_date'])

# Outlier Detection & Removal via Interquartile Range (IQR)
Q1 = df['revenue'].quantile(0.25)
Q3 = df['revenue'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_cleaned = df[(df['revenue'] >= lower_bound) & (df['revenue'] <= upper_bound)]
print(f"Cleaned dataset shape after removing outliers: {df_cleaned.shape}")

# Export Cleaned Data
df_cleaned.to_csv('../data/cleaned_sales_data.csv', index=False)
print("Cleaned data exported to week_1_analytics/data/cleaned_sales_data.csv")

# 3. KPI Calculation & Statistical Summaries
print("\n=== Key Performance Indicators ===")
print(f"Total Revenue: ${df_cleaned['revenue'].sum():,.2f}")
print(f"Average Order Value: ${df_cleaned['revenue'].mean():,.2f}")
print(f"Median Order Value: ${df_cleaned['revenue'].median():,.2f}")
print(f"Revenue Standard Deviation: ${df_cleaned['revenue'].std():,.2f}")

# 4. Data Visualization
plt.figure(figsize=(10, 5))
sns.histplot(df_cleaned['revenue'], kde=True, color='teal')
plt.title('Distribution of Cleaned Transaction Revenue')
plt.xlabel('Revenue ($)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('../reports/revenue_distribution.png')
print("Visualization saved to week_1_analytics/reports/revenue_distribution.png")